# 🏆 Premier League Match Predictor
## Análisis de Datos y Predicción con Machine Learning

**Objetivo**: Predecir resultados de partidos de la Premier League utilizando análisis de datos y modelos de Machine Learning.

**Dataset**: Datos en tiempo real de football-data.org API

---

## 📦 SECCIÓN 1: Instalación de Dependencias

Instalamos todas las librerías necesarias para el proyecto.

In [ ]:
# Instalar dependencias (descomentar si estás en Colab)
# !pip install -q pandas numpy matplotlib seaborn scikit-learn requests gradio fastapi uvicorn pyngrok nest-asyncio

print("✅ Dependencias instaladas correctamente")

## 📚 SECCIÓN 2: Importación de Librerías

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc
from sklearn.preprocessing import StandardScaler

# API y utilidades
import requests
import json
import pickle
import time
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Librerías importadas correctamente")

## 🌐 SECCIÓN 3: Configuración y Obtención de Datos

Conectamos a la API de football-data.org para obtener datos actuales de la Premier League.

In [ ]:
# Configuración de API
API_KEY = "f50c3bb69922405b8963e15c66c23877"
LEAGUE = "PL"  # Premier League
API_BASE = "https://api.football-data.org/v4"
HEADERS = {"X-Auth-Token": API_KEY}

# Crear directorios si no existen
os.makedirs('data', exist_ok=True)
os.makedirs('outputs', exist_ok=True)
os.makedirs('models', exist_ok=True)

print("✅ Configuración completada")

In [ ]:
def fetch_standings():
    """
    Obtiene los datos actuales de la tabla de posiciones de la Premier League.
    """
    url = f"{API_BASE}/competitions/{LEAGUE}/standings"
    
    try:
        print("📡 Conectando a la API...")
        response = requests.get(url, headers=HEADERS, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            table = data["standings"][0]["table"]
            
            teams = []
            for row in table:
                teams.append({
                    "id": row["team"]["id"],
                    "name": row["team"]["name"],
                    "position": row["position"],
                    "played": row["playedGames"],
                    "won": row["won"],
                    "draw": row["draw"],
                    "lost": row["lost"],
                    "points": row["points"],
                    "goalsFor": row["goalsFor"],
                    "goalsAgainst": row["goalsAgainst"],
                    "goalDifference": row["goalDifference"]
                })
            
            print(f"✅ Datos obtenidos exitosamente: {len(teams)} equipos")
            return pd.DataFrame(teams)
        
        elif response.status_code == 429:
            print("⚠️ Límite de API alcanzado. Intentando cargar datos guardados...")
            if os.path.exists('data/teams_raw.csv'):
                return pd.read_csv('data/teams_raw.csv')
            else:
                raise Exception("No hay datos guardados disponibles")
        
        else:
            raise Exception(f"Error en API: {response.status_code}")
    
    except Exception as e:
        print(f"❌ Error: {e}")
        if os.path.exists('data/teams_raw.csv'):
            print("📂 Cargando datos guardados previamente...")
            return pd.read_csv('data/teams_raw.csv')
        raise

# Obtener datos
df_teams = fetch_standings()

# Guardar datos crudos
df_teams.to_csv('data/teams_raw.csv', index=False)
print(f"\n💾 Datos guardados en 'data/teams_raw.csv'")

# Mostrar primeras filas
print("\n📊 Primeras filas del dataset:")
df_teams.head()

## 🔍 SECCIÓN 4: Análisis Exploratorio de Datos (EDA)

Exploramos los datos para entender patrones, distribuciones y relaciones entre variables.

In [ ]:
# Información general del dataset
print("="*60)
print("INFORMACIÓN DEL DATASET")
print("="*60)
print(f"\nNúmero de equipos: {len(df_teams)}")
print(f"Número de variables: {len(df_teams.columns)}")
print(f"\nColumnas: {list(df_teams.columns)}")

print("\n" + "="*60)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("="*60)
df_teams.describe()

In [ ]:
# Verificar valores nulos
print("\nValores nulos por columna:")
print(df_teams.isnull().sum())

# Tipos de datos
print("\nTipos de datos:")
print(df_teams.dtypes)

### 📊 Visualización 1: Tabla de Posiciones Actual

In [ ]:
# Mostrar tabla completa ordenada por posición
table_display = df_teams[['position', 'name', 'played', 'won', 'draw', 'lost', 'goalsFor', 'goalsAgainst', 'goalDifference', 'points']]
table_display = table_display.sort_values('position')

print("\n" + "="*100)
print("TABLA DE POSICIONES - PREMIER LEAGUE")
print("="*100)
print(table_display.to_string(index=False))

### 📊 Visualización 2: Distribución de Puntos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Puntos por equipo
df_sorted = df_teams.sort_values('points', ascending=True)
axes[0].barh(df_sorted['name'], df_sorted['points'], color='steelblue')
axes[0].set_xlabel('Puntos', fontsize=12)
axes[0].set_title('Puntos por Equipo', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Gráfico 2: Distribución de puntos
axes[1].hist(df_teams['points'], bins=10, color='coral', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Puntos', fontsize=12)
axes[1].set_ylabel('Frecuencia', fontsize=12)
axes[1].set_title('Distribución de Puntos', fontsize=14, fontweight='bold')
axes[1].axvline(df_teams['points'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df_teams["points"].mean():.1f}')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/01_distribucion_puntos.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: outputs/01_distribucion_puntos.png")

### 📊 Visualización 3: Ataque vs Defensa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 10 mejores ataques
top_attack = df_teams.nlargest(10, 'goalsFor').sort_values('goalsFor')
axes[0].barh(top_attack['name'], top_attack['goalsFor'], color='green', alpha=0.7)
axes[0].set_xlabel('Goles a Favor', fontsize=12)
axes[0].set_title('Top 10 Mejores Ataques', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Top 10 mejores defensas (menos goles en contra)
top_defense = df_teams.nsmallest(10, 'goalsAgainst').sort_values('goalsAgainst', ascending=False)
axes[1].barh(top_defense['name'], top_defense['goalsAgainst'], color='red', alpha=0.7)
axes[1].set_xlabel('Goles en Contra', fontsize=12)
axes[1].set_title('Top 10 Mejores Defensas (Menos Goles en Contra)', fontsize=14, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/02_ataque_defensa.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: outputs/02_ataque_defensa.png")

### 📊 Visualización 4: Scatter Plot - Goles a Favor vs Puntos

In [ ]:
plt.figure(figsize=(12, 8))

# Scatter plot
scatter = plt.scatter(df_teams['goalsFor'], df_teams['points'], 
                     s=df_teams['won']*30,  # tamaño basado en victorias
                     c=df_teams['position'],  # color basado en posición
                     cmap='RdYlGn_r',  # colormap invertido
                     alpha=0.6, 
                     edgecolors='black',
                     linewidth=1.5)

# Etiquetas de equipos
for idx, row in df_teams.iterrows():
    plt.annotate(row['name'], 
                (row['goalsFor'], row['points']),
                fontsize=8,
                alpha=0.7,
                xytext=(5, 5),
                textcoords='offset points')

plt.colorbar(scatter, label='Posición en la Tabla')
plt.xlabel('Goles a Favor', fontsize=12)
plt.ylabel('Puntos', fontsize=12)
plt.title('Relación entre Goles a Favor y Puntos\n(Tamaño = Victorias, Color = Posición)', 
         fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/03_scatter_goles_puntos.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: outputs/03_scatter_goles_puntos.png")

### 📊 Visualización 5: Matriz de Correlación

In [ ]:
# Seleccionar solo columnas numéricas relevantes
numeric_cols = ['played', 'won', 'draw', 'lost', 'points', 'goalsFor', 'goalsAgainst', 'goalDifference']
correlation_matrix = df_teams[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, 
           annot=True, 
           fmt='.2f', 
           cmap='coolwarm', 
           center=0,
           square=True,
           linewidths=1,
           cbar_kws={"shrink": 0.8})

plt.title('Matriz de Correlación de Variables', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('outputs/04_matriz_correlacion.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: outputs/04_matriz_correlacion.png")
print("\n📊 Correlaciones más fuertes con 'points':")
print(correlation_matrix['points'].sort_values(ascending=False))

### 📊 Visualización 6: Resultados (Victorias, Empates, Derrotas)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

# Datos
x = np.arange(len(df_teams))
width = 0.25

df_sorted = df_teams.sort_values('position')

# Barras
bars1 = ax.bar(x - width, df_sorted['won'], width, label='Victorias', color='green', alpha=0.8)
bars2 = ax.bar(x, df_sorted['draw'], width, label='Empates', color='orange', alpha=0.8)
bars3 = ax.bar(x + width, df_sorted['lost'], width, label='Derrotas', color='red', alpha=0.8)

ax.set_xlabel('Equipos', fontsize=12)
ax.set_ylabel('Número de Partidos', fontsize=12)
ax.set_title('Distribución de Resultados por Equipo', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_sorted['name'], rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/05_resultados_equipos.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: outputs/05_resultados_equipos.png")

## ⚙️ SECCIÓN 5: Feature Engineering

Creamos nuevas variables (features) que ayudarán a mejorar las predicciones del modelo.

In [ ]:
# Crear copia del dataframe
df_features = df_teams.copy()

# 1. Métricas por partido
df_features['points_per_game'] = df_features['points'] / df_features['played'].replace(0, 1)
df_features['goals_for_per_game'] = df_features['goalsFor'] / df_features['played'].replace(0, 1)
df_features['goals_against_per_game'] = df_features['goalsAgainst'] / df_features['played'].replace(0, 1)

# 2. Fuerza de ataque y defensa
df_features['attack_strength'] = df_features['goals_for_per_game']
df_features['defense_strength'] = 1 / (df_features['goals_against_per_game'].replace(0, 0.1))  # Inverso, mayor es mejor

# 3. Eficiencia
df_features['win_rate'] = (df_features['won'] / df_features['played'].replace(0, 1)) * 100
df_features['draw_rate'] = (df_features['draw'] / df_features['played'].replace(0, 1)) * 100
df_features['loss_rate'] = (df_features['lost'] / df_features['played'].replace(0, 1)) * 100

# 4. Balance ofensivo-defensivo
df_features['goal_difference_per_game'] = df_features['goalDifference'] / df_features['played'].replace(0, 1)

# 5. Score de forma (basado en puntos y diferencia de goles)
df_features['form_score'] = (
    df_features['points_per_game'] * 0.6 + 
    df_features['goal_difference_per_game'] * 0.4
)

# 6. Score total del equipo (heurística mejorada)
df_features['team_score'] = (
    df_features['attack_strength'] * 0.35 +
    df_features['defense_strength'] * 0.35 +
    df_features['form_score'] * 0.30
)

# 7. Categoría de rendimiento
def categorize_performance(points_per_game):
    if points_per_game >= 2.0:
        return 'Excelente'
    elif points_per_game >= 1.5:
        return 'Bueno'
    elif points_per_game >= 1.0:
        return 'Regular'
    else:
        return 'Malo'

df_features['performance_category'] = df_features['points_per_game'].apply(categorize_performance)

print("✅ Features creados exitosamente")
print(f"\nNúmero total de features: {len(df_features.columns)}")
print(f"\nNuevas features creadas:")
new_features = ['points_per_game', 'goals_for_per_game', 'goals_against_per_game', 
               'attack_strength', 'defense_strength', 'win_rate', 'draw_rate', 'loss_rate',
               'goal_difference_per_game', 'form_score', 'team_score', 'performance_category']
for feature in new_features:
    print(f"  - {feature}")

# Guardar dataset con features
df_features.to_csv('data/teams_with_features.csv', index=False)
print("\n💾 Dataset con features guardado: data/teams_with_features.csv")

In [ ]:
# Mostrar estadísticas de los nuevos features
print("\n📊 Estadísticas de Features Principales:\n")
print(df_features[['name', 'points_per_game', 'attack_strength', 'defense_strength', 
                   'form_score', 'team_score', 'performance_category']].sort_values('team_score', ascending=False))

### 📊 Visualización 7: Comparación de Features Principales

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Attack Strength
df_sorted = df_features.sort_values('attack_strength', ascending=True)
axes[0, 0].barh(df_sorted['name'], df_sorted['attack_strength'], color='green', alpha=0.7)
axes[0, 0].set_xlabel('Fuerza de Ataque (Goles/Partido)', fontsize=10)
axes[0, 0].set_title('Fuerza de Ataque por Equipo', fontsize=12, fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# 2. Defense Strength
df_sorted = df_features.sort_values('defense_strength', ascending=True)
axes[0, 1].barh(df_sorted['name'], df_sorted['defense_strength'], color='red', alpha=0.7)
axes[0, 1].set_xlabel('Fuerza Defensiva (1/Goles en contra)', fontsize=10)
axes[0, 1].set_title('Fuerza Defensiva por Equipo', fontsize=12, fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)

# 3. Form Score
df_sorted = df_features.sort_values('form_score', ascending=True)
axes[1, 0].barh(df_sorted['name'], df_sorted['form_score'], color='blue', alpha=0.7)
axes[1, 0].set_xlabel('Score de Forma', fontsize=10)
axes[1, 0].set_title('Forma Actual por Equipo', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# 4. Team Score (Overall)
df_sorted = df_features.sort_values('team_score', ascending=True)
axes[1, 1].barh(df_sorted['name'], df_sorted['team_score'], color='purple', alpha=0.7)
axes[1, 1].set_xlabel('Score Total del Equipo', fontsize=10)
axes[1, 1].set_title('Ranking General por Score', fontsize=12, fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/06_features_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: outputs/06_features_comparison.png")

## 🤖 SECCIÓN 6: Preparación de Datos para Machine Learning

Creamos un dataset de enfrentamientos simulados para entrenar el modelo.

In [ ]:
def create_match_dataset(df):
    """
    Crea un dataset de enfrentamientos entre equipos.
    Para cada par de equipos, genera features comparativas y un resultado basado en stats.
    """
    matches = []
    
    # Generar todos los posibles enfrentamientos
    for i, team_a in df.iterrows():
        for j, team_b in df.iterrows():
            if i != j:  # No enfrentar un equipo consigo mismo
                # Features del enfrentamiento
                match = {
                    'teamA_id': team_a['id'],
                    'teamB_id': team_b['id'],
                    'teamA_name': team_a['name'],
                    'teamB_name': team_b['name'],
                    
                    # Diferencias en stats
                    'attack_diff': team_a['attack_strength'] - team_b['attack_strength'],
                    'defense_diff': team_a['defense_strength'] - team_b['defense_strength'],
                    'form_diff': team_a['form_score'] - team_b['form_score'],
                    'points_diff': team_a['points'] - team_b['points'],
                    'goal_diff_diff': team_a['goalDifference'] - team_b['goalDifference'],
                    
                    # Stats absolutas de cada equipo
                    'teamA_attack': team_a['attack_strength'],
                    'teamA_defense': team_a['defense_strength'],
                    'teamA_form': team_a['form_score'],
                    'teamB_attack': team_b['attack_strength'],
                    'teamB_defense': team_b['defense_strength'],
                    'teamB_form': team_b['form_score'],
                    
                    # Score total
                    'teamA_score': team_a['team_score'],
                    'teamB_score': team_b['team_score'],
                    'score_diff': team_a['team_score'] - team_b['team_score'],
                }
                
                # Target: 1 si gana team_a, 0 si gana team_b, basado en team_score
                # Agregamos algo de aleatoriedad para simular incertidumbre
                prob_a_wins = 1 / (1 + np.exp(-match['score_diff']))  # Sigmoid
                match['winner'] = 1 if prob_a_wins > 0.5 else 0
                
                matches.append(match)
    
    return pd.DataFrame(matches)

# Crear dataset de enfrentamientos
print("🔄 Generando dataset de enfrentamientos...")
df_matches = create_match_dataset(df_features)

print(f"\n✅ Dataset creado: {len(df_matches)} enfrentamientos")
print(f"   - Distribución de ganadores:")
print(f"     Team A gana: {(df_matches['winner'] == 1).sum()}")
print(f"     Team B gana: {(df_matches['winner'] == 0).sum()}")

# Guardar dataset
df_matches.to_csv('data/matches_dataset.csv', index=False)
print("\n💾 Dataset guardado: data/matches_dataset.csv")

# Mostrar ejemplos
print("\n📋 Ejemplos de enfrentamientos:")
df_matches[['teamA_name', 'teamB_name', 'score_diff', 'winner']].head(10)

## 🧠 SECCIÓN 7: Entrenamiento de Modelos de Machine Learning

Entrenamos y comparamos diferentes modelos para predecir el resultado de los partidos.

In [ ]:
# Seleccionar features para el modelo
feature_columns = [
    'attack_diff', 'defense_diff', 'form_diff', 'points_diff', 'goal_diff_diff',
    'teamA_attack', 'teamA_defense', 'teamA_form',
    'teamB_attack', 'teamB_defense', 'teamB_form',
    'score_diff'
]

X = df_matches[feature_columns]
y = df_matches['winner']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(f"📊 Dataset dividido:")
print(f"   Training: {len(X_train)} muestras")
print(f"   Testing: {len(X_test)} muestras")
print(f"   Features: {len(feature_columns)}")

In [ ]:
# Normalización de datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Guardar scaler
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Datos normalizados")
print("💾 Scaler guardado: models/scaler.pkl")

In [ ]:
# Diccionario para almacenar modelos y resultados
models = {}
results = {}

print("🤖 Entrenando modelos...\n")
print("="*80)

# 1. Regresión Logística (Baseline)
print("\n1️⃣ Regresión Logística (Baseline)")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_pred)
models['Logistic Regression'] = lr_model
results['Logistic Regression'] = lr_accuracy
print(f"   Accuracy: {lr_accuracy:.4f}")

# 2. Random Forest
print("\n2️⃣ Random Forest")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)
models['Random Forest'] = rf_model
results['Random Forest'] = rf_accuracy
print(f"   Accuracy: {rf_accuracy:.4f}")

# 3. Gradient Boosting
print("\n3️⃣ Gradient Boosting")
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=5)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_pred)
models['Gradient Boosting'] = gb_model
results['Gradient Boosting'] = gb_accuracy
print(f"   Accuracy: {gb_accuracy:.4f}")

print("\n" + "="*80)
print("\n✅ Entrenamiento completado\n")

# Comparación de modelos
print("📊 COMPARACIÓN DE MODELOS:\n")
results_df = pd.DataFrame(list(results.items()), columns=['Modelo', 'Accuracy'])
results_df = results_df.sort_values('Accuracy', ascending=False)
print(results_df.to_string(index=False))

# Identificar mejor modelo
best_model_name = results_df.iloc[0]['Modelo']
best_model = models[best_model_name]
print(f"\n🏆 Mejor modelo: {best_model_name} (Accuracy: {results_df.iloc[0]['Accuracy']:.4f})")

# Guardar mejor modelo
with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("💾 Mejor modelo guardado: models/best_model.pkl")

### 📊 Visualización 8: Comparación de Modelos

In [ ]:
plt.figure(figsize=(10, 6))

colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
bars = plt.bar(results_df['Modelo'], results_df['Accuracy'], color=colors, alpha=0.8, edgecolor='black')

# Añadir valores sobre las barras
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.xlabel('Modelo', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Comparación de Accuracy entre Modelos', fontsize=14, fontweight='bold')
plt.ylim([0, 1.1])
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/07_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: outputs/07_model_comparison.png")

## 📈 SECCIÓN 8: Evaluación Detallada del Mejor Modelo

In [ ]:
# Usar el mejor modelo para predicciones
if best_model_name == 'Logistic Regression':
    y_pred = lr_pred
    y_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
elif best_model_name == 'Random Forest':
    y_pred = rf_pred
    y_pred_proba = rf_model.predict_proba(X_test)[:, 1]
else:  # Gradient Boosting
    y_pred = gb_pred
    y_pred_proba = gb_model.predict_proba(X_test)[:, 1]

# Classification Report
print("📊 CLASSIFICATION REPORT:")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['Team B Wins', 'Team A Wins']))

### 📊 Visualización 9: Matriz de Confusión

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
           xticklabels=['Team B Wins', 'Team A Wins'],
           yticklabels=['Team B Wins', 'Team A Wins'],
           cbar_kws={'label': 'Cantidad'})

plt.xlabel('Predicción', fontsize=12)
plt.ylabel('Real', fontsize=12)
plt.title(f'Matriz de Confusión - {best_model_name}', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/08_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: outputs/08_confusion_matrix.png")

### 📊 Visualización 10: Curva ROC

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title(f'Receiver Operating Characteristic (ROC) - {best_model_name}', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/09_roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Gráfico guardado: outputs/09_roc_curve.png")
print(f"\n📊 AUC Score: {roc_auc:.4f}")

### 📊 Visualización 11: Feature Importance (Solo para Random Forest / Gradient Boosting)

In [ ]:
if best_model_name in ['Random Forest', 'Gradient Boosting']:
    # Importancia de features
    feature_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(12, 8))
    plt.barh(feature_importance['feature'], feature_importance['importance'], color='teal', alpha=0.7)
    plt.xlabel('Importancia', fontsize=12)
    plt.title(f'Importancia de Features - {best_model_name}', fontsize=14, fontweight='bold')
    plt.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('outputs/10_feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Gráfico guardado: outputs/10_feature_importance.png")
    print("\n📊 Top 5 Features más importantes:")
    print(feature_importance.head())
else:
    print("ℹ️ Feature importance solo disponible para Random Forest y Gradient Boosting")

## 🎯 SECCIÓN 9: Sistema de Predicción

Implementamos la función de predicción que usaremos para predecir resultados de partidos.

In [ ]:
def predict_match_ml(team_a_id, team_b_id, model, scaler, df_features, use_scaling=False):
    """
    Predice el resultado de un partido usando el modelo ML entrenado.
    
    Args:
        team_a_id: ID del equipo A
        team_b_id: ID del equipo B
        model: Modelo ML entrenado
        scaler: Scaler para normalización (puede ser None)
        df_features: DataFrame con features de equipos
        use_scaling: Si usar normalización (True para Logistic Regression)
    
    Returns:
        dict con predicción detallada
    """
    # Obtener datos de equipos
    try:
        team_a = df_features[df_features['id'] == team_a_id].iloc[0]
        team_b = df_features[df_features['id'] == team_b_id].iloc[0]
    except IndexError:
        return {"error": "Equipo no encontrado"}
    
    # Crear features del enfrentamiento
    match_features = pd.DataFrame([{
        'attack_diff': team_a['attack_strength'] - team_b['attack_strength'],
        'defense_diff': team_a['defense_strength'] - team_b['defense_strength'],
        'form_diff': team_a['form_score'] - team_b['form_score'],
        'points_diff': team_a['points'] - team_b['points'],
        'goal_diff_diff': team_a['goalDifference'] - team_b['goalDifference'],
        'teamA_attack': team_a['attack_strength'],
        'teamA_defense': team_a['defense_strength'],
        'teamA_form': team_a['form_score'],
        'teamB_attack': team_b['attack_strength'],
        'teamB_defense': team_b['defense_strength'],
        'teamB_form': team_b['form_score'],
        'score_diff': team_a['team_score'] - team_b['team_score'],
    }])
    
    # Normalizar si es necesario
    if use_scaling and scaler is not None:
        match_features_scaled = scaler.transform(match_features)
        prediction_proba = model.predict_proba(match_features_scaled)[0]
    else:
        prediction_proba = model.predict_proba(match_features)[0]
    
    # Probabilidades
    prob_b_wins = prediction_proba[0] * 100
    prob_a_wins = prediction_proba[1] * 100
    
    # Determinar ganador
    if prob_a_wins > prob_b_wins:
        winner = team_a['name']
        confidence = prob_a_wins
    else:
        winner = team_b['name']
        confidence = prob_b_wins
    
    return {
        "teamA": team_a['name'],
        "teamB": team_b['name'],
        "probA": round(prob_a_wins, 2),
        "probB": round(prob_b_wins, 2),
        "winner": winner,
        "confidence": round(confidence, 2),
        "model_used": best_model_name,
        "teamA_stats": {
            "attack": round(team_a['attack_strength'], 2),
            "defense": round(team_a['defense_strength'], 2),
            "form": round(team_a['form_score'], 2),
            "overall_score": round(team_a['team_score'], 2)
        },
        "teamB_stats": {
            "attack": round(team_b['attack_strength'], 2),
            "defense": round(team_b['defense_strength'], 2),
            "form": round(team_b['form_score'], 2),
            "overall_score": round(team_b['team_score'], 2)
        }
    }

print("✅ Función de predicción creada")

### 🧪 Prueba del Sistema de Predicción

In [ ]:
# Ejemplo de predicción
print("🧪 PRUEBA DEL SISTEMA DE PREDICCIÓN\n")
print("="*80)

# Seleccionar dos equipos aleatorios
team_a = df_features.iloc[0]
team_b = df_features.iloc[1]

print(f"\n🔵 Equipo A: {team_a['name']}")
print(f"🔴 Equipo B: {team_b['name']}")

# Hacer predicción
use_scaling = (best_model_name == 'Logistic Regression')
prediction = predict_match_ml(team_a['id'], team_b['id'], best_model, scaler, df_features, use_scaling)

print("\n" + "="*80)
print("RESULTADO DE LA PREDICCIÓN")
print("="*80)
print(f"\n🏆 Ganador predicho: {prediction['winner']}")
print(f"📊 Confianza: {prediction['confidence']}%")
print(f"\nProbabilidades:")
print(f"  {prediction['teamA']}: {prediction['probA']}%")
print(f"  {prediction['teamB']}: {prediction['probB']}%")
print(f"\n🤖 Modelo utilizado: {prediction['model_used']}")

print("\n" + "="*80)
print("ESTADÍSTICAS DE LOS EQUIPOS")
print("="*80)
print(f"\n{prediction['teamA']}:")
for key, value in prediction['teamA_stats'].items():
    print(f"  {key}: {value}")

print(f"\n{prediction['teamB']}:")
for key, value in prediction['teamB_stats'].items():
    print(f"  {key}: {value}")

## 🌐 SECCIÓN 10: Interfaz Interactiva con Gradio

Creamos una interfaz web simple para realizar predicciones de forma interactiva.

In [ ]:
# Instalar Gradio si no está instalado
try:
    import gradio as gr
except ImportError:
    print("Instalando Gradio...")
    !pip install -q gradio
    import gradio as gr

print("✅ Gradio importado correctamente")

In [ ]:
def gradio_predict(team_a_name, team_b_name):
    """
    Función wrapper para Gradio que acepta nombres de equipos.
    """
    try:
        # Obtener IDs de equipos por nombre
        team_a = df_features[df_features['name'] == team_a_name]
        team_b = df_features[df_features['name'] == team_b_name]
        
        if team_a.empty or team_b.empty:
            return "❌ Equipo no encontrado. Por favor verifica los nombres."
        
        if team_a_name == team_b_name:
            return "❌ Debes seleccionar dos equipos diferentes."
        
        team_a_id = team_a.iloc[0]['id']
        team_b_id = team_b.iloc[0]['id']
        
        # Hacer predicción
        use_scaling = (best_model_name == 'Logistic Regression')
        result = predict_match_ml(team_a_id, team_b_id, best_model, scaler, df_features, use_scaling)
        
        # Formatear resultado
        output = f"""
## 🏆 PREDICCIÓN DE PARTIDO

### {result['teamA']} 🆚 {result['teamB']}

---

**🎯 Ganador Predicho:** {result['winner']}

**📊 Confianza:** {result['confidence']}%

### Probabilidades:
- **{result['teamA']}:** {result['probA']}%
- **{result['teamB']}:** {result['probB']}%

---

### 📈 Estadísticas de {result['teamA']}:
- Ataque: {result['teamA_stats']['attack']}
- Defensa: {result['teamA_stats']['defense']}
- Forma: {result['teamA_stats']['form']}
- Score General: {result['teamA_stats']['overall_score']}

### 📉 Estadísticas de {result['teamB']}:
- Ataque: {result['teamB_stats']['attack']}
- Defensa: {result['teamB_stats']['defense']}
- Forma: {result['teamB_stats']['form']}
- Score General: {result['teamB_stats']['overall_score']}

---

🤖 **Modelo:** {result['model_used']}

⚠️ *Esta es una predicción basada en datos estadísticos. No es una recomendación de apuesta.*
        """
        
        return output
    
    except Exception as e:
        return f"❌ Error: {str(e)}"

# Obtener lista de equipos
team_names = sorted(df_features['name'].tolist())

# Crear interfaz Gradio
iface = gr.Interface(
    fn=gradio_predict,
    inputs=[
        gr.Dropdown(choices=team_names, label="🔵 Equipo A"),
        gr.Dropdown(choices=team_names, label="🔴 Equipo B")
    ],
    outputs=gr.Markdown(label="Resultado de la Predicción"),
    title="⚽ Premier League Match Predictor",
    description="Selecciona dos equipos para predecir el resultado del partido usando Machine Learning.",
    theme="soft",
    examples=[
        [team_names[0], team_names[1]],
        [team_names[2], team_names[3]],
    ]
)

print("✅ Interfaz Gradio creada")
print("\n🚀 Lanzando interfaz...")

In [ ]:
# Lanzar interfaz (share=True para URL pública temporal)
iface.launch(share=True, debug=True)

## 💾 SECCIÓN 11: Exportación de CSVs Finales

Exportamos todos los datos procesados y resultados en formato CSV.

In [ ]:
print("💾 EXPORTANDO ARCHIVOS CSV...\n")
print("="*80)

# 1. Teams con features
output_teams = df_features[['id', 'name', 'position', 'played', 'won', 'draw', 'lost', 
                           'points', 'goalsFor', 'goalsAgainst', 'goalDifference',
                           'attack_strength', 'defense_strength', 'form_score', 
                           'team_score', 'performance_category']]
output_teams.to_csv('outputs/teams_analysis.csv', index=False)
print("✅ 1. teams_analysis.csv - Análisis completo de equipos")

# 2. Match predictions (ejemplo de algunas predicciones)
predictions_list = []
for i in range(min(50, len(df_features))):
    for j in range(i+1, min(50, len(df_features))):
        team_a = df_features.iloc[i]
        team_b = df_features.iloc[j]
        use_scaling = (best_model_name == 'Logistic Regression')
        pred = predict_match_ml(team_a['id'], team_b['id'], best_model, scaler, df_features, use_scaling)
        predictions_list.append(pred)

df_predictions = pd.DataFrame(predictions_list)
df_predictions.to_csv('outputs/match_predictions.csv', index=False)
print(f"✅ 2. match_predictions.csv - {len(df_predictions)} predicciones de partidos")

# 3. Model performance metrics
model_metrics = pd.DataFrame({
    'model': list(results.keys()),
    'accuracy': list(results.values())
})
model_metrics['best_model'] = model_metrics['model'] == best_model_name
model_metrics.to_csv('outputs/model_performance.csv', index=False)
print("✅ 3. model_performance.csv - Métricas de rendimiento de modelos")

# 4. Training dataset completo
df_matches.to_csv('outputs/training_dataset.csv', index=False)
print(f"✅ 4. training_dataset.csv - Dataset de entrenamiento ({len(df_matches)} enfrentamientos)")

# 5. Summary report
summary = {
    'metric': [
        'Total Teams',
        'Total Matches in Training',
        'Best Model',
        'Best Model Accuracy',
        'Total Features',
        'Training Samples',
        'Test Samples',
        'AUC Score'
    ],
    'value': [
        len(df_features),
        len(df_matches),
        best_model_name,
        f"{results[best_model_name]:.4f}",
        len(feature_columns),
        len(X_train),
        len(X_test),
        f"{roc_auc:.4f}"
    ]
}
df_summary = pd.DataFrame(summary)
df_summary.to_csv('outputs/project_summary.csv', index=False)
print("✅ 5. project_summary.csv - Resumen del proyecto")

print("\n" + "="*80)
print("\n🎉 TODOS LOS ARCHIVOS CSV EXPORTADOS EXITOSAMENTE")
print("\n📂 Ubicación: carpeta 'outputs/'")
print("\n" + "="*80)

## 📊 SECCIÓN 12: Resumen Final del Proyecto

In [ ]:
print("\n" + "="*80)
print("" * 30 + "RESUMEN FINAL" + " " * 30)
print("="*80 + "\n")

print("📁 ARCHIVOS GENERADOS:\n")
print("📂 data/")
print("   ├── teams_raw.csv")
print("   ├── teams_with_features.csv")
print("   └── matches_dataset.csv")
print("\n📂 outputs/")
print("   ├── teams_analysis.csv")
print("   ├── match_predictions.csv")
print("   ├── model_performance.csv")
print("   ├── training_dataset.csv")
print("   ├── project_summary.csv")
print("   └── [10 gráficos PNG]")
print("\n📂 models/")
print("   ├── best_model.pkl")
print("   └── scaler.pkl")

print("\n" + "="*80)
print("\n📊 ESTADÍSTICAS DEL PROYECTO:\n")
print(f"   🏆 Equipos analizados: {len(df_features)}")
print(f"   🤖 Mejor modelo: {best_model_name}")
print(f"   🎯 Accuracy: {results[best_model_name]:.4f}")
print(f"   📈 AUC Score: {roc_auc:.4f}")
print(f"   🔢 Features utilizados: {len(feature_columns)}")
print(f"   📝 Predicciones generadas: {len(df_predictions)}")
print(f"   📊 Gráficos creados: 10")

print("\n" + "="*80)
print("\n✅ PROYECTO COMPLETADO EXITOSAMENTE")
print("\n" + "="*80)

print("\n🎯 PRÓXIMOS PASOS:\n")
print("   1. Revisar las visualizaciones en la carpeta 'outputs/'")
print("   2. Analizar los CSVs generados")
print("   3. Probar la interfaz Gradio para predicciones interactivas")
print("   4. Ajustar hiperparámetros del modelo si es necesario")
print("   5. Recolectar más datos históricos para mejorar predicciones")
print("\n" + "="*80)